In [ ]:
!pip install transformers accelerate torch pandas openpyxl

In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# Set the model name
model_name = "Qwen/Qwen2.5-3B-Instruct"

print("Loading the model and tokenizer... Please wait.")

# Load the tokenizer to convert text into a format the model understands
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the model directly onto the GPU (device_map="auto") for faster processing
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded successfully!")

Loading the model and tokenizer... Please wait.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:

import torch
import pandas as pd

def generate_advice(raw_tip):
    if pd.isna(raw_tip) or not str(raw_tip).strip():
        return ""

    prompt = f"""Rewrite the energy-saving tip as one concise, neutral,
informational statement in natural English.

Requirements:
- State the information, condition, or fact; do not give an instruction.
- Never begin with command verbs such as Use, Set, Install, Lower,
  Turn off, Keep, Place, Clean, or Drain.
- Do not address the reader directly.
- Preserve the original meaning, numbers, and important conditions.
- Do not invent facts or benefits.
- Output only one complete sentence.

Examples:
Original: Turn off ceiling fans when you leave a room.
Rewritten: Ceiling fans can be turned off when a room is unoccupied.

Original: Install tight-fitting curtains or shades on windows to retain heat.
Rewritten: Tight-fitting curtains or shades on windows help retain heat.

Original: Set your water heater temperature to 120°F.
Rewritten: A water heater temperature of 120°F is recommended.

Original: {str(raw_tip).strip()}
Rewritten:"""

    messages = [
        {
            "role": "system",
            "content": (
                "You rewrite energy-saving tips as neutral informational "
                "statements, never as commands. Follow the examples."
            )
        },
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    input_device = model.get_input_embeddings().weight.device

    if input_device.type == "meta":
        raise RuntimeError(
            "Input embeddings are on the meta device. "
            "Check how the model was loaded."
        )

    model_inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(input_device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=60,
            do_sample=False
        )

    new_tokens = generated_ids[
        0, model_inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    # Remove common quotation marks
    response = response.strip("\"'“”‘’")
    return " ".join(response.split())

In [ ]:
print("Generating advice for all tips...")

# Enable progress bar for pandas apply function
tqdm.pandas()

# Apply the function to each row in the 'raw tips' column and store in a new 'advice' column
df['advice'] = df['raw tips'].progress_apply(generate_advice)

# Overwrite the original Excel file with the updated dataset
output_file = "EcowattTips.xlsx"
df.to_excel(output_file, index=False)

print(f"Process completed successfully! The file '{output_file}' has been updated and overwritten.")

Generating advice for all tips...


100%|██████████| 55/55 [14:53<00:00, 16.25s/it]

Process completed successfully! The file 'EcowattTips.xlsx' has been updated and overwritten.
